# 04 — Datasets & DataLoaders

`Dataset` defines how to access a single sample; `DataLoader` handles batching, shuffling, and parallel loading.

## Built-in TensorDataset

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

X = torch.randn(100, 10)
y = torch.randint(0, 2, (100,))

dataset = TensorDataset(X, y)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

for xb, yb in loader:
    print(xb.shape, yb.shape)
    break


## Custom Dataset

Subclass `Dataset` and implement `__len__` and `__getitem__`.

In [ ]:
from torch.utils.data import Dataset

class CSVDataset(Dataset):
    def __init__(self, features, labels, transform=None):
        self.features = features
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        x = self.features[idx]
        y = self.labels[idx]
        if self.transform:
            x = self.transform(x)
        return x, y

ds = CSVDataset(X, y)
len(ds), ds[0]


## DataLoader Options

In [ ]:
loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,       # parallel data loading processes
    pin_memory=True,     # faster host-to-GPU transfer
    drop_last=True,      # drop incomplete final batch
)


## Train/Val Split

In [ ]:
from torch.utils.data import random_split

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)


## torchvision Datasets & Transforms

In [ ]:
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# folder structure: root/class_x/xxx.png, root/class_y/yyy.png
img_dataset = datasets.ImageFolder(root="data/train", transform=transform)

# built-in benchmark datasets (auto-download)
mnist = datasets.MNIST(root="data", train=True, download=True, transform=transforms.ToTensor())
cifar = datasets.CIFAR10(root="data", train=True, download=True, transform=transform)


## Custom Image Dataset (reading from disk)

In [ ]:
from PIL import Image
import os

class ImageDataset(Dataset):
    def __init__(self, image_dir, labels, transform=None):
        self.image_dir = image_dir
        self.filenames = os.listdir(image_dir)
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        path = os.path.join(self.image_dir, self.filenames[idx])
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = self.labels[idx]
        return image, label


## Custom collate_fn

Useful for variable-length sequences (e.g. text/NLP) where samples can't be stacked directly.

In [ ]:
def collate_fn(batch):
    xs, ys = zip(*batch)
    xs_padded = torch.nn.utils.rnn.pad_sequence(xs, batch_first=True)
    ys = torch.tensor(ys)
    return xs_padded, ys

loader = DataLoader(dataset, batch_size=16, collate_fn=collate_fn)


## IterableDataset (for streaming / very large data)

In [ ]:
from torch.utils.data import IterableDataset

class StreamDataset(IterableDataset):
    def __init__(self, source):
        self.source = source

    def __iter__(self):
        for item in self.source:
            yield item

stream_loader = DataLoader(StreamDataset(range(1000)), batch_size=32)


## Common Gotchas
- `num_workers > 0` on macOS/Windows requires code to run under `if __name__ == '__main__':` in scripts (not an issue in notebooks).
- Forgetting `shuffle=True` on the training loader hurts convergence.
- Normalization stats must match what a pretrained model expects when doing transfer learning.
- `drop_last=True` avoids batch-size-1 batches that can break BatchNorm.